# BERT + GRU, 2-class sentiment (Kaggle)

Kaggle-ready reproduction of `Codes/2-class/BERT_w_GRU.ipynb` from *Sentiment analysis in Bengali via transfer learning using multi-lingual BERT* (ICCIT 2020). The paper reports **71% accuracy** for BERT_BSA + GRU on the 2-class task (Table V).

This notebook sources preprocessing/model code from `https://github.com/habibour/sent` and reads the raw 3-class dataset from `/kaggle/input/datasets/reversedthoutgts/bangla-dataset/{train_.csv,test_.csv}` (same data as this repo's `Dataset/train.csv` / `Dataset/test.csv`). It then:
1. Cleans the text and drops the Neutral class to recreate the paper's 2-class dataset (13,120 rows total).
2. Trains the same `BERTGRUSentiment` architecture/hyperparameters as the original notebook.
3. Evaluates on the held-out test set and compares the result to the paper's 71%.

To smoke-test the pipeline before committing to the full run, lower `N_EPOCHS` in the training-config cell below.

In [ ]:
import os

REPO_DIR = '/kaggle/working/sent'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/habibour/sent.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(REPO_DIR)

In [ ]:
!pip install -q transformers bnlp_toolkit bangla-stemmer

import nltk
nltk.download('punkt')

In [ ]:
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel

from src.model import (
    SentimentDataset, make_collate_fn, BERTGRUSentiment, freeze_bert,
    count_parameters, train_epoch, evaluate, epoch_time, train_valid_split,
    set_seed, SEED, BATCH_SIZE, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS,
    BIDIRECTIONAL, DROPOUT, LEARNING_RATE, WEIGHT_DECAY, N_EPOCHS,
    MAX_INPUT_LENGTH, BERT_MODEL_NAME,
)
from src.preprocess import run_pipeline

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Training config

Defaults match the original notebook exactly. Lower `N_EPOCHS` here for a quick smoke test.

In [ ]:
TRAIN_PATH = '/kaggle/input/datasets/reversedthoutgts/bangla-dataset/train_.csv'
TEST_PATH = '/kaggle/input/datasets/reversedthoutgts/bangla-dataset/test_.csv'
CHECKPOINT_PATH = '/kaggle/working/BERT_GRU_2class.pt'
PAPER_ACCURACY = 0.71

# N_EPOCHS = 20  # imported default from src.model; override here to smoke-test, e.g. N_EPOCHS = 2
print('N_EPOCHS:', N_EPOCHS)

In [ ]:
train_df, test_df = run_pipeline(TRAIN_PATH, TEST_PATH, apply_stemming=True)
train_df, valid_df = train_valid_split(train_df, split_ratio=0.85, seed=SEED)

print(f"Number of training examples: {len(train_df)}")
print(f"Number of validation examples: {len(valid_df)}")
print(f"Number of testing examples: {len(test_df)}")

In [ ]:
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)

train_dataset = SentimentDataset(train_df['Data'], train_df['label'], tokenizer, MAX_INPUT_LENGTH)
valid_dataset = SentimentDataset(valid_df['Data'], valid_df['label'], tokenizer, MAX_INPUT_LENGTH)
test_dataset = SentimentDataset(test_df['Data'], test_df['label'], tokenizer, MAX_INPUT_LENGTH)

collate_fn = make_collate_fn(tokenizer.pad_token_id)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
bert = BertModel.from_pretrained(BERT_MODEL_NAME)

model = BERTGRUSentiment(bert, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, BIDIRECTIONAL, DROPOUT)
freeze_bert(model)
model = model.to(device)

print(f'The model has {count_parameters(model):,} trainable parameters')

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss().to(device)

In [ ]:
best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):
    start_time = time.time()

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    valid_loss, valid_acc, _, _ = evaluate(model, valid_loader, criterion, device)

    end_time = time.time()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)

    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))

test_loss, test_acc, all_predictions, targets = evaluate(model, test_loader, criterion, device)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')
print(f"\nPaper's reported BERT_BSA+GRU 2-class accuracy: {PAPER_ACCURACY*100:.2f}%")
print(f"This run's test accuracy: {test_acc*100:.2f}%")
print(f"Difference: {(test_acc - PAPER_ACCURACY)*100:+.2f} percentage points")